In [1]:
import psycopg2
#import sqlalchemy as sq
import pandas as pd
import numpy as np
import mariadb
#import dask.dataframe as dd
import json
import os
import shutil
import subprocess
from pathlib import Path
import pyodbc
# import rpy2.robjects as robjects
# from rpy2.robjects import pandas2ri
import requests

In [2]:
# !pip install psycopg2 pandas mariadb pyodbc 
# !pip ins

In [3]:
# r = robjects.r
# r['source']('D://Cropnuts/DSML147/calccec_v2.R')
# calcCec = robjects.globalenv['calcCec']

In [4]:
import mariadb
import sys

username = "scoring"
password = "idkltb93e0eomejp"
host = 'spectral-msql-jul-24-backup-do-user-2276924-0.b.db.ondigitalocean.com'
port = 25060
database="farmlabv3_live"
# schema="farmlabv3_live"
# schema="historical"
def get_db_cursor():
    try:
        conn = mariadb.connect(
            user=username,
            password=password,
            host=host,
            port=port,
            database=database
    
        )
        return conn
    except mariadb.Error as e:
        print(f"Error connecting to MariaDB Platform: {e}")
        sys.exit(1)

conn = get_db_cursor()
cur = conn.cursor()

In [5]:
conn_lims = pyodbc.connect("Driver={SQL Server};"
                            "Server=192.168.5.18\CROPNUT;"
                            "Database=cropnuts;"
                            "uid=thomasTsuma;pwd=GR^KX$uRe9#JwLc6")
cursor_lims = conn_lims.cursor()

In [6]:
# crops = pd.read_sql("SELECT id, name FROM crop",con=conn).set_index("id")['name'].to_dict()

In [7]:
# crops

In [8]:
# crops[0] = None

In [9]:
comparison_df = pd.read_csv("inputFiles/IIES Test Protocol.xlsx - Sheet1.csv")    

In [10]:
comparison_df

,barcode,client_name,farm_name,crop,country_name,ph,phosphorus,potassium,Organic Matter,Yield Target,SOIL CORRECTION,SOIL CORRECTION.1,SOIL CORRECTION:MANURE/COMPOST,PLANTING,PLANTING.1,PLANTING.2,PLANTING.3,TOP DRESS,TOP DRESS.1,TOP DRESS.2
0,TEST-DS3-0001,Institute for International Economic Studies,test,Maize,Uganda,high,high,high,high,5,NaN,NaN,NaN,NP (23.23),NaN,NaN,NaN,NaN,UREA,NaN
1,TEST-DS3-0002,Institute for International Economic Studies,test,Maize,Uganda,optimum,high,optimum,optimum,5,NaN,NaN,5000.0,NP (23.23),NaN,NaN,NaN,NaN,UREA,NaN
2,TEST-DS3-0003,Institute for International Economic Studies,test,Maize,Uganda,low,high,low,low,5,calcitic lime,dolomitic lime,5000.0,NaN,NPK (17.17.17),NaN,NaN,CAN,NaN,MOP
3,TEST-DS3-0004,Institute for International Economic Studies,test,Maize,Uganda,high,optimum,high,high,5,NaN,NaN,5000.0,NaN,NaN,DAP,NaN,NaN,UREA,NaN
4,TEST-DS3-0005,Institute for International Economic Studies,test,Maize,Uganda,optimum,optimum,optimum,optimum,5,NaN,NaN,5000.0,NaN,NaN,NaN,NPK 12.24.12 +5S,NaN,UREA,NaN
5,TEST-DS3-0006,Institute for International Economic Studies,test,Maize,Uganda,low,optimum,low,low,5,calcitic lime,dolomitic lime,5000.0,NaN,NaN,NaN,NPK 12.24.12 +5S,CAN,NaN,MOP
6,TEST-DS3-0007,Institute for International Economic Studies,test,Maize,Uganda,high,low,high,high,5,NaN,NaN,5000.0,NaN,NaN,DAP,NaN,NaN,UREA,NaN
7,TEST-DS3-0008,Institute for International Economic Studies,test,Maize,Uganda,optimum,low,optimum,optimum,5,NaN,NaN,5000.0,NaN,NaN,NaN,NPK 12.24.12 +5S,NaN,UREA,NaN
8,TEST-DS3-0009,Institute for International Economic Studies,test,Maize,Uganda,low,low,low,low,5,calcitic lime,dolomitic lime,5000.0,NaN,NaN,NaN,NPK 12.24.12 +5S,CAN,NaN,MOP


In [11]:
scores = pd.read_csv("scoring_output_2023-08-01.csv")

In [12]:
# report_data = pd.read_csv("report_data_client.csv")

In [13]:
# test_df = report_data.query('barcode.str.contains("TEST-DS")', engine='python')

In [14]:
# report_data = report_data.loc[~(report_data['Unnamed: 0'].isin(test_df['Unnamed: 0']))]

In [15]:
# report_data['crop_id'] = [str(i).split("-")[-1].strip() for i in report_data['barcode'].values.tolist()]
# report_data['crop_id'] = report_data['crop_id'].replace("nan", 0)
# report_data['crop_id'] = report_data['crop_id'].fillna(0)

In [16]:
# report_data['barcode'] = report_data['barcode'].apply(lambda x : "-".join(str(x).split("-")[:-1]) if len(str(x).split("-")) == 4 else "")

In [17]:
# crop_names = []
# for i in report_data['crop_id']:
#     try:
#         crop_names.append(crops[int(i)])
#     except Exception as e:
#         crop_names.append(i)
# report_data['crop_name'] = crop_names

In [18]:
# len(crop_names)

In [19]:
# report_data.loc[(report_data['crop_name'] == 'Maize') & report_data['client_name'] == 'Kalro']

In [20]:
scores

,Unnamed: 0.1,Unnamed: 0,spectral_sample_id,sample_date,spectral_batch_id,id,spectral_sample_id.1,spectral_batch_id.1,lab_id,phosphorus,...,acid_saturation,hydrogen_percent,other_bases,cecCalcul,calciumPercSat,magnesiumPercSat,potassiumPercSat,sodiumPercSat,CaMgRatio,CNRatio
0,0,1,8150,2023-11-06 06:12:03,277.0,222528,8150,277,11.0,NaN,...,0.45,0.000000,-10.578607,45.378522,85.0,23.0,2.40,1.10,6.25,16.106142
1,1,2,8268,2023-11-01 02:50:03,280.0,222606,8268,280,11.0,NaN,...,1.80,0.000000,-5.985627,29.104419,82.0,23.0,0.84,0.20,6.07,14.611598
2,2,3,8348,2023-11-01 02:39:43,322.0,222644,8348,322,11.0,NaN,...,0.61,0.000000,-7.171522,39.027836,84.0,21.0,2.30,0.65,6.80,14.371103
3,3,4,9904,2023-11-01 02:36:25,364.0,222969,9904,364,11.0,NaN,...,1.20,0.000000,-7.143825,22.019080,79.0,25.0,2.50,0.84,5.31,14.237278
4,8,9,162994,2023-10-08 11:23:25,5040.0,461930,162994,5040,22.0,low,...,0.26,2.874999,4.591667,11.209137,66.0,21.0,3.60,2.30,5.20,15.765128
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30243,43811,43812,253653,2024-06-23 14:36:12,9220.0,931692,253653,9220,11.0,very low,...,2.10,10.167582,5.077839,4.125453,68.0,11.0,1.10,4.50,9.88,14.341612
30244,43812,43813,253654,2024-06-23 14:40:30,9220.0,931691,253654,9220,11.0,very low,...,1.10,2.885849,4.592390,4.749112,73.0,13.0,1.90,4.50,8.99,16.716230
30245,43813,43814,253655,2024-06-23 14:43:58,9220.0,931690,253655,9220,11.0,very low,...,2.10,12.195740,5.213049,5.688500,65.0,14.0,0.61,3.10,7.67,17.597024
30246,43819,43820,253711,2024-06-24 10:17:37,9232.0,931869,253711,9232,39.0,very low,...,6.40,11.927576,5.195172,1.966576,54.0,20.0,0.38,9.00,4.54,6.519935


In [21]:
# report_data

In [22]:
scores[['cecCalcul']]

,cecCalcul
0,45.378522
1,29.104419
2,39.027836
3,22.019080
4,11.209137
...,...
30243,4.125453
30244,4.749112
30245,5.688500
30246,1.966576


In [23]:
config = pd.read_sql("""SELECT [Chemical_Config_Id]
       ,[chemical_code]
      ,[Spectral_Lod]
      ,[spectral_Decimal_places]
      ,[spectral_Significant_figure]
  FROM [cropnuts].[dbo].[Chemicals_Config]
  where [Type_Code]=4""",con=conn_lims)
chemicals = pd.read_sql("SELECT chemical_code, chemical_name FROM Chemicals",con=conn_lims)
config = pd.merge(config, chemicals,on="chemical_code",how="inner")
config.chemical_name = [i.lower().replace(" ","_").replace("(","").replace(")","").replace(".","") for i in config.chemical_name]
config= config[['chemical_name','spectral_Decimal_places','spectral_Significant_figure']]
config = config.set_index('chemical_name')
config = config.to_dict()

C:\Users\tsuma.thomas\AppData\Local\Temp\ipykernel_18412\2924685361.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  config = pd.read_sql("""SELECT [Chemical_Config_Id]
C:\Users\tsuma.thomas\AppData\Local\Temp\ipykernel_18412\2924685361.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  chemicals = pd.read_sql("SELECT chemical_code, chemical_name FROM Chemicals",con=conn_lims)


In [24]:
def round_number(number, precision):
    if precision == 0.1:
        return round(number, 1)
    elif precision == 0.01:
        return round(number, 2)
    elif precision == 0.001:
        return round(number, 3)
    elif precision == 0.0001:
        return round(number, 4)
    else:
        return number

In [25]:
print(round_number(12345.6789, 0.01))


12345.68


In [26]:
# config

In [27]:
for column in scores.columns:
    if column in config['spectral_Decimal_places'].keys() and column != "phosphorus":
        if (config['spectral_Significant_figure'][column]) >= 0:
            scores[column] = [ round_number(i,config['spectral_Significant_figure'][column]) for i in scores[column] ]
            # round(decimals=int(config['spectral_Decimal_places'][column]))
        elif (config['spectral_Decimal_places'][column]) >= 0:
            scores[column] = scores[column].round(decimals=int(config['spectral_Decimal_places'][column]))
        else:
            continue


In [28]:
_ = scores.copy(deep=True)

In [29]:
scores.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'spectral_sample_id', 'sample_date',
       'spectral_batch_id', 'id', 'spectral_sample_id.1',
       'spectral_batch_id.1', 'lab_id', 'phosphorus', 'texture', 'aluminium',
       'boron', 'calcium', 'clay', 'copper', 'ec_salts',
       'exchangeable_acidity', 'iron', 'magnesium', 'manganese',
       'organic_carbon', 'ph', 'phosphorus_sorption_index', 'potassium',
       'sand', 'silt', 'sodium', 'sulphur', 'total_nitrogen', 'zinc',
       'cn_ratio', 'acid_saturation', 'hydrogen_percent', 'other_bases',
       'cecCalcul', 'calciumPercSat', 'magnesiumPercSat', 'potassiumPercSat',
       'sodiumPercSat', 'CaMgRatio', 'CNRatio'],
      dtype='object')

In [30]:
scores[['potassium','potassiumPercSat']]

,potassium,potassiumPercSat
0,417.497,2.40
1,94.969,0.84
2,343.689,2.30
3,211.000,2.50
4,158.373,3.60
...,...,...
30243,17.535,1.10
30244,34.291,1.90
30245,13.636,0.61
30246,2.900,0.38


In [31]:
12.67899 * 0.01

0.1267899

In [32]:
# _['ph'] = scores[['ph','cecCalcul']].iloc[0:3].apply(lambda row: classifyResults(row, chemical="ph", lim1=10, lim2=20, lim3=30, crop_code=47, client_id=85593, chemical_code=3), axis=1)

In [33]:
scores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30248 entries, 0 to 30247
Data columns (total 42 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Unnamed: 0.1               30248 non-null  int64  
 1   Unnamed: 0                 30248 non-null  int64  
 2   spectral_sample_id         30248 non-null  int64  
 3   sample_date                30248 non-null  object 
 4   spectral_batch_id          30240 non-null  float64
 5   id                         30248 non-null  int64  
 6   spectral_sample_id.1       30248 non-null  int64  
 7   spectral_batch_id.1        30248 non-null  int64  
 8   lab_id                     30200 non-null  float64
 9   phosphorus                 30244 non-null  object 
 10  texture                    30200 non-null  object 
 11  aluminium                  4 non-null      float64
 12  boron                      30200 non-null  float64
 13  calcium                    30248 non-null  flo

In [34]:
scores = scores.dropna(subset = ['cecCalcul'])

In [35]:
def classifyResults(row, chemical, lim1, lim2, lim3, crop_code, client_id, chemical_code):   
    X = row[chemical]
    cec = row['cecCalcul']
     
    if chemical == 'potassium':
        potassium_veryhigh_guide, potassium_high_guide, potassium_sample_guide, potassium_low_guide, potassium_verylow_guide = cursor_lims.execute(
        f"""
            DECLARE @Critical_K FLOAT;
            select @Critical_K = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial K Level%' and Client_id={client_id} and Lab_Code= 0;
        
            DECLARE @Critical_Ca FLOAT;
            select @Critical_Ca = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial Ca Level%' and Client_id={client_id} and Lab_Code= 0;
        
            DECLARE @Critical_Mg FLOAT;
            select @Critical_Mg = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial Mg Level%' and Client_id={client_id} and Lab_Code= 0;
        
            DECLARE @interpretation_Code INT;
            select @interpretation_Code=interpretation_code from interpretations WHERE crop_code = {crop_code} AND Client_Id = {client_id} and Chemical_Code = {chemical_code};
        
            DECLARE @veryhigh_guide FLOAT;
            select @veryhigh_guide=dbo.Sample_Guides(@interpretation_Code, 0);
            select @veryhigh_guide = {cec} * @veryhigh_guide/100 *390;
        
            DECLARE @high_guide FLOAT;
            select @high_guide=dbo.Sample_Guides(@interpretation_Code, 1);
            select @high_guide = {cec} * @high_guide/100 *390;
            
            DECLARE @sample_guide FLOAT;
            select @sample_guide=dbo.Sample_Guides(@interpretation_Code, 2)
            select @sample_guide= {cec} *@sample_guide/100 *390
        
            DECLARE @low_guide FLOAT;
            select @low_guide=dbo.Sample_Guides(@interpretation_Code, 3)
            select @low_guide= {cec} * @low_guide/100 * 390
        
            DECLARE @verylow_guide FLOAT;
            select @verylow_guide=dbo.Sample_Guides(@interpretation_Code, 4)
            select @verylow_guide= {cec} * @verylow_guide/100 * 390
            
            select @veryhigh_guide, @high_guide,  @sample_guide, @low_guide, @verylow_guide
        """).fetchone()
        if X <= potassium_verylow_guide:
            return "very low"
        elif X <= potassium_low_guide:
            return "low"
        elif X <= potassium_high_guide:
            return "optimum"
        else:
            return "high"
    
    elif chemical == 'magnesium':
        magnesium_veryhigh_guide, magnesium_high_guide, magnesium_sample_guide, magnesium_low_guide, magnesium_verylow_guide = cursor_lims.execute(
        f"""
        DECLARE @Critical_K FLOAT;
        select @Critical_K = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial K Level%' and Client_id={client_id} and Lab_Code= 0;
    
        DECLARE @Critical_Ca FLOAT;
        select @Critical_Ca = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial Ca Level%' and Client_id={client_id} and Lab_Code= 0;
    
        DECLARE @Critical_Mg FLOAT;
        select @Critical_Mg = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial Mg Level%' and Client_id={client_id} and Lab_Code= 0;
    
        DECLARE @interpretation_Code INT;
        select @interpretation_Code=interpretation_code from interpretations WHERE crop_code = {crop_code} AND Client_Id = {client_id} and Chemical_Code = {chemical_code};
    
        DECLARE @veryhigh_guide FLOAT;
        select @veryhigh_guide=dbo.Sample_Guides(@interpretation_Code, 0);
        select @veryhigh_guide = {cec} * @veryhigh_guide/100 *390;
    
        DECLARE @high_guide FLOAT;
        select @high_guide=dbo.Sample_Guides(@interpretation_Code, 1);
        select @high_guide = {cec} * @high_guide/100 *390;
        
        DECLARE @sample_guide FLOAT;
        select @sample_guide=dbo.Sample_Guides(@interpretation_Code, 2)
        select @sample_guide= {cec} *@sample_guide/100 *390
    
        DECLARE @low_guide FLOAT;
        select @low_guide=dbo.Sample_Guides(@interpretation_Code, 3)
        select @low_guide= {cec} * @low_guide/100 * 390
    
        DECLARE @verylow_guide FLOAT;
        select @verylow_guide=dbo.Sample_Guides(@interpretation_Code, 4)
        select @verylow_guide= {cec} * @verylow_guide/100 * 390
        
        select @veryhigh_guide, @high_guide,  @sample_guide, @low_guide, @verylow_guide
        """).fetchone()
        if X <= magnesium_verylow_guide:
            return "very low"
        elif X <= magnesium_low_guide:
            return "low"
        elif X <= magnesium_high_guide:
            return "optimum"
        else:
            return "high"
    elif chemical == 'calcium':
        calcium_veryhigh_guide, calcium_high_guide, calcium_sample_guide, calcium_low_guide, calcium_verylow_guide = cursor_lims.execute(
        f"""
        DECLARE @Critical_K FLOAT;
        select @Critical_K = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial K Level%' and Client_id={client_id} and Lab_Code= 0;
    
        DECLARE @Critical_Ca FLOAT;
        select @Critical_Ca = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial Ca Level%' and Client_id={client_id} and Lab_Code= 0;
    
        DECLARE @Critical_Mg FLOAT;
        select @Critical_Mg = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial Mg Level%' and Client_id={client_id} and Lab_Code= 0;
    
        DECLARE @interpretation_Code INT;
        select @interpretation_Code=interpretation_code from interpretations WHERE crop_code = {crop_code} AND Client_Id = {client_id} and Chemical_Code = {chemical_code};
    
        DECLARE @veryhigh_guide FLOAT;
        select @veryhigh_guide=dbo.Sample_Guides(@interpretation_Code, 0);
        select @veryhigh_guide = {cec} * @veryhigh_guide/100 *390;
    
        DECLARE @high_guide FLOAT;
        select @high_guide=dbo.Sample_Guides(@interpretation_Code, 1);
        select @high_guide = {cec} * @high_guide/100 *390;
        
        DECLARE @sample_guide FLOAT;
        select @sample_guide=dbo.Sample_Guides(@interpretation_Code, 2)
        select @sample_guide= {cec} *@sample_guide/100 *390
    
        DECLARE @low_guide FLOAT;
        select @low_guide=dbo.Sample_Guides(@interpretation_Code, 3)
        select @low_guide= {cec} * @low_guide/100 * 390
    
        DECLARE @verylow_guide FLOAT;
        select @verylow_guide=dbo.Sample_Guides(@interpretation_Code, 4)
        select @verylow_guide= {cec} * @verylow_guide/100 * 390
        
        select @veryhigh_guide, @high_guide,  @sample_guide, @low_guide, @verylow_guide
        """).fetchone()
        if X <= calcium_verylow_guide:
            return "very low"
        elif X <= calcium_low_guide:
            return "low"
        elif X <= calcium_high_guide:
            return "optimum"
        else:
            return "high"
    else:
        if X <= lim1:
            return "very low"
        elif X <= lim2:
            return "low"
        elif X <= lim3:
            return "optimum"
        else:
            return "high"

In [36]:
def convertGuides(row, chemical, verylow, low, optimum, high, veryhigh, crop_code, client_id, chemical_code):   
    cec = row['cecCalcul']
    if chemical == 'potassium':
        try:
            veryhigh_guide, high_guide, sample_guide, low_guide, verylow_guide = cursor_lims.execute(
            f"""
                DECLARE @Critical_K FLOAT;
                select @Critical_K = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial K Level%' and Client_id={client_id} and Lab_Code= 0;
            
                DECLARE @Critical_Ca FLOAT;
                select @Critical_Ca = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial Ca Level%' and Client_id={client_id} and Lab_Code= 0;
            
                DECLARE @Critical_Mg FLOAT;
                select @Critical_Mg = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial Mg Level%' and Client_id={client_id} and Lab_Code= 0;
            
                DECLARE @interpretation_Code INT;
                select @interpretation_Code=interpretation_code from interpretations WHERE crop_code = {crop_code} AND Client_Id = {client_id} and Chemical_Code = {chemical_code};
            
                DECLARE @veryhigh_guide FLOAT;
                select @veryhigh_guide=dbo.Sample_Guides(@interpretation_Code, 0);
                select @veryhigh_guide = {cec} * @veryhigh_guide/100 *390;
            
                DECLARE @high_guide FLOAT;
                select @high_guide=dbo.Sample_Guides(@interpretation_Code, 1);
                select @high_guide = {cec} * @high_guide/100 *390;
                
                DECLARE @sample_guide FLOAT;
                select @sample_guide=dbo.Sample_Guides(@interpretation_Code, 2)
                select @sample_guide= {cec} *@sample_guide/100 *390
            
                DECLARE @low_guide FLOAT;
                select @low_guide=dbo.Sample_Guides(@interpretation_Code, 3)
                select @low_guide= {cec} * @low_guide/100 * 390
            
                DECLARE @verylow_guide FLOAT;
                select @verylow_guide=dbo.Sample_Guides(@interpretation_Code, 4)
                select @verylow_guide= {cec} * @verylow_guide/100 * 390
                
                select @veryhigh_guide, @high_guide,  @sample_guide, @low_guide, @verylow_guide
            """).fetchone()
        except Exception as e:
            
            print(f"""
                DECLARE @Critical_K FLOAT;
                select @Critical_K = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial K Level%' and Client_id={client_id} and Lab_Code= 0;
            
                DECLARE @Critical_Ca FLOAT;
                select @Critical_Ca = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial Ca Level%' and Client_id={client_id} and Lab_Code= 0;
            
                DECLARE @Critical_Mg FLOAT;
                select @Critical_Mg = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial Mg Level%' and Client_id={client_id} and Lab_Code= 0;
            
                DECLARE @interpretation_Code INT;
                select @interpretation_Code=interpretation_code from interpretations WHERE crop_code = {crop_code} AND Client_Id = {client_id} and Chemical_Code = {chemical_code};
            
                DECLARE @veryhigh_guide FLOAT;
                select @veryhigh_guide=dbo.Sample_Guides(@interpretation_Code, 0);
                select @veryhigh_guide = {cec} * @veryhigh_guide/100 *390;
            
                DECLARE @high_guide FLOAT;
                select @high_guide=dbo.Sample_Guides(@interpretation_Code, 1);
                select @high_guide = {cec} * @high_guide/100 *390;
                
                DECLARE @sample_guide FLOAT;
                select @sample_guide=dbo.Sample_Guides(@interpretation_Code, 2)
                select @sample_guide= {cec} *@sample_guide/100 *390
            
                DECLARE @low_guide FLOAT;
                select @low_guide=dbo.Sample_Guides(@interpretation_Code, 3)
                select @low_guide= {cec} * @low_guide/100 * 390
            
                DECLARE @verylow_guide FLOAT;
                select @verylow_guide=dbo.Sample_Guides(@interpretation_Code, 4)
                select @verylow_guide= {cec} * @verylow_guide/100 * 390
                
                select @veryhigh_guide, @high_guide,  @sample_guide, @low_guide, @verylow_guide
            """)
            raise("There was an issue, ",e)
        return pd.DataFrame({"crop": crop_code, "client": client, "chemical": chemical_code, "guides": [verylow_guide,low_guide,sample_guide,high_guide,veryhigh_guide], "status":["very low","low","optimum","high","very high"]})
    elif chemical == 'magnesium':
        veryhigh_guide, high_guide, sample_guide, low_guide, verylow_guide = cursor_lims.execute(
        f"""
        DECLARE @Critical_K FLOAT;
        select @Critical_K = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial K Level%' and Client_id={client_id} and Lab_Code= 0;
    
        DECLARE @Critical_Ca FLOAT;
        select @Critical_Ca = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial Ca Level%' and Client_id={client_id} and Lab_Code= 0;
    
        DECLARE @Critical_Mg FLOAT;
        select @Critical_Mg = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial Mg Level%' and Client_id={client_id} and Lab_Code= 0;
    
        DECLARE @interpretation_Code INT;
        select @interpretation_Code=interpretation_code from interpretations WHERE crop_code = {crop_code} AND Client_Id = {client_id} and Chemical_Code = {chemical_code};
    
        DECLARE @veryhigh_guide FLOAT;
        select @veryhigh_guide=dbo.Sample_Guides(@interpretation_Code, 0);
        select @veryhigh_guide = {cec} * @veryhigh_guide/100 *390;
    
        DECLARE @high_guide FLOAT;
        select @high_guide=dbo.Sample_Guides(@interpretation_Code, 1);
        select @high_guide = {cec} * @high_guide/100 *390;
        
        DECLARE @sample_guide FLOAT;
        select @sample_guide=dbo.Sample_Guides(@interpretation_Code, 2)
        select @sample_guide= {cec} *@sample_guide/100 *390
    
        DECLARE @low_guide FLOAT;
        select @low_guide=dbo.Sample_Guides(@interpretation_Code, 3)
        select @low_guide= {cec} * @low_guide/100 * 390
    
        DECLARE @verylow_guide FLOAT;
        select @verylow_guide=dbo.Sample_Guides(@interpretation_Code, 4)
        select @verylow_guide= {cec} * @verylow_guide/100 * 390
        
        select @veryhigh_guide, @high_guide,  @sample_guide, @low_guide, @verylow_guide
        """).fetchone()
        return pd.DataFrame({"crop": crop_code, "client": client, "chemical": chemical_code, "guides": [verylow_guide,low_guide,sample_guide,high_guide,veryhigh_guide], "status":["very low","low","optimum","high","very high"]})
    elif chemical == 'calcium':
        veryhigh_guide, high_guide, sample_guide, low_guide, verylow_guide = cursor_lims.execute(
        f"""
        DECLARE @Critical_K FLOAT;
        select @Critical_K = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial K Level%' and Client_id={client_id} and Lab_Code= 0;
    
        DECLARE @Critical_Ca FLOAT;
        select @Critical_Ca = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial Ca Level%' and Client_id={client_id} and Lab_Code= 0;
    
        DECLARE @Critical_Mg FLOAT;
        select @Critical_Mg = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial Mg Level%' and Client_id={client_id} and Lab_Code= 0;
    
        DECLARE @interpretation_Code INT;
        select @interpretation_Code=interpretation_code from interpretations WHERE crop_code = {crop_code} AND Client_Id = {client_id} and Chemical_Code = {chemical_code};
    
        DECLARE @veryhigh_guide FLOAT;
        select @veryhigh_guide=dbo.Sample_Guides(@interpretation_Code, 0);
        select @veryhigh_guide = {cec} * @veryhigh_guide/100 *390;
    
        DECLARE @high_guide FLOAT;
        select @high_guide=dbo.Sample_Guides(@interpretation_Code, 1);
        select @high_guide = {cec} * @high_guide/100 *390;
        
        DECLARE @sample_guide FLOAT;
        select @sample_guide=dbo.Sample_Guides(@interpretation_Code, 2)
        select @sample_guide= {cec} *@sample_guide/100 *390
    
        DECLARE @low_guide FLOAT;
        select @low_guide=dbo.Sample_Guides(@interpretation_Code, 3)
        select @low_guide= {cec} * @low_guide/100 * 390
    
        DECLARE @verylow_guide FLOAT;
        select @verylow_guide=dbo.Sample_Guides(@interpretation_Code, 4)
        select @verylow_guide= {cec} * @verylow_guide/100 * 390
        
        select @veryhigh_guide, @high_guide,  @sample_guide, @low_guide, @verylow_guide
        """).fetchone()
        return pd.DataFrame({"crop": crop_code, "client": client, "chemical": chemical_code, "guides": [verylow_guide,low_guide,sample_guide,high_guide,veryhigh_guide], "status":["very low","low","optimum","high","very high"]})
    else:
        return pd.DataFrame({"crop": crop_code, "client": client, "chemical": chemical_code, "guides": [verylow, low, optimum, high, veryhigh], "status":["very low","low","optimum","high","very high"]})

In [37]:
scores = scores.drop_duplicates(subset="spectral_sample_id")

In [38]:
scores['organic_matter'] = scores['organic_carbon'] * 1.72

In [39]:
def getCoords(location):
    url = 'https://nominatim.openstreetmap.org/search'
    
        # Parameters for the API request
    params = {
        'q': location,
        'format': 'json',    # Output format
        'addressdetails': 1, # Include a breakdown of the address
        'limit': 1           # Limit to 1 result (best match)
    }
    
    # Send GET request to Nominatim API
    response = requests.get(url, params=params)
    print(response.json())
    return response.json()[0]['lat'], response.json()[0]['lon']

In [40]:
scores[['potassium','potassiumPercSat']]

,potassium,potassiumPercSat
0,417.497,2.40
1,94.969,0.84
2,343.689,2.30
3,211.000,2.50
4,158.373,3.60
...,...,...
30243,17.535,1.10
30244,34.291,1.90
30245,13.636,0.61
30246,2.900,0.38


In [41]:
spectral_sample_df = pd.DataFrame()
count = 0

In [42]:
# renamed_opus_folder = "opus_renamed"
# opus_folder = "opus"
# spectral_sample_output_file = "Spectral Sample Output.csv"
# location = "Kenya"
# # lat, lng = getCoords(location)
# renaming = {}

# os.makedirs(f"outputFiles/{opus_folder}",exist_ok=True)
# os.makedirs(f"outputFiles/{renamed_opus_folder}",exist_ok=True)


# aez_df = pd.read_csv("inputFiles/2024-08-06 Nigeria AEZ coordinates.csv")
# aez_df = aez_df.reset_index()
# for index, row in comparison_df.iterrows():
#     count+=1
#     subfolder = int(count/100)
#     os.makedirs(f"outputFiles/{opus_folder}/{subfolder}",exist_ok=True)
#     os.makedirs(f"outputFiles/{renamed_opus_folder}/{subfolder}",exist_ok=True)
#     barcode = row['barcode'].strip()
#     crop = row['crop'].strip()
#     client = row['client_name'].strip()
#     farm = row['farm_name'].strip()
#     ph = str(row['ph']).strip().lower()
#     phosphorus = str(row['phosphorus']).strip().lower()
#     potassium = str(row['potassium']).strip().lower()
#     organic_matter = str(row['Organic Matter']).strip().lower()
#     #calcium = str(row['calcium']).strip()
#     # magnesium = str(row['magnesium']).strip()
#     print(barcode)

#     # folder = f"./outputFiles/{crop}_ph-{ph}_phosphorus-{phosphorus}_potassium-{potassium}_organicmatter-{organic_matter}_calcium-{calcium}_magnesium-{magnesium}"
#     #folder = f"./outputFiles/{crop.replace(' ','')}_ph-{ph}_phosphorus-{phosphorus}_potassium-{potassium}_organicmatter-{organic_matter}_calcium-{calcium}"
#     folder = f"./outputFiles/{crop.replace(' ','')}_ph-{ph}_phosphorus-{phosphorus}_potassium-{potassium}_organicmatter-{organic_matter}"
#     if(folder in os.listdir("./outputFiles")):
#         continue


#     crop_reports = report_data.copy(deep=True)
    
#     crop_reports = crop_reports.loc[(crop_reports['crop_name'].str.lower() == crop.lower().strip()) & (crop_reports['client_name'].str.lower() == client.lower())]
#     if(len(crop_reports) == 0):
#         print(client, ": ", crop)
#         continue
#     if(phosphorus in ['low','very low','optimum','high','very high']):
#         crop_reports = crop_reports.loc[crop_reports['Available P'] == phosphorus]
#         print("phosphorus class: ",phosphorus)
#         print("phosphorus",len(crop_reports))
#     if(potassium in ['low','very low','optimum','high','very high']):
#         crop_reports = crop_reports.loc[crop_reports['Exchangeable K'] == potassium]
#         print("potassium class: ",potassium)
#         print("potassium",len(crop_reports))
#     if(ph in ['low','very low','optimum','high','very high']):
#         crop_reports = crop_reports.loc[crop_reports['pH'] == ph]
#         print("ph class: ",ph)
#         print("ph",len(crop_reports))
#     #if(calcium in ['low','very low','optimum','high','very high']):
#      #   crop_reports = crop_reports.loc[crop_reports['calcium_classes'] == calcium]
#       #  print("calcium class: ",calcium)
#       #  print("calcium",len(crop_reports))
#     if(organic_matter in ['low','very low','optimum','high','very high']):
#         crop_reports = crop_reports.loc[crop_reports['Organic Matter'] == organic_matter]
#         print("organic matter class: ",organic_matter)
#         print("organic_matter",len(crop_reports))
#     # if(magnesium in ['low','very low','optimum','high','very high']):
#     #     crop_reports = crop_reports.loc[crop_reports['magnesium'] == magnesium]
#     #     print("magnesium",len(crop_reports))


    
#     if(len(crop_reports) == 0):
#         continue
    
#     spectral_df = pd.read_sql(f"""
#         SELECT 
#         spectral_sample_id, 
#         barcode AS 'Barcode', 
#         farmer_name AS 'Name of Farmer', 
#         phone_number AS 'Phone Number',
#         sampler_name AS 'Sampler Name', 
#         spectral_batch_id, 
#         longitude AS 'Longitude',
#         latitude AS 'Latitude',
#         sample_date AS 'Sample Date', 
#         analysis_type AS 'Analysis Name', 
#         tree_population AS 'Tree_Population(Total in Field)',  
#         TIMESTAMPDIFF(YEAR,'2024-05-28',date_of_planting) AS 'Tree Age(Years)', 
#         field_size AS 	'Field Size (Acre)'
#         FROM SpectralSample
#         WHERE spectral_sample_id IN {str([j for j in crop_reports['spectral_sample_id'].values]).replace("[","(").replace("]",")")} 
#         AND spectral_batch_id > 3355
#     """, con=conn)
#     # scoring_res = pd.read_sql("
#     for index, record in spectral_df.iterrows():
#         farmlab_barcode = record['Barcode']
#         batch_no = record['spectral_batch_id']
#         batch_no = str(batch_no).split('.')[0]
        
#         if len([i for i in Path(f'./outputFiles/{renamed_opus_folder}/{subfolder}').rglob(f"*{barcode}*")]) == 2:
#             break
#         #try:
#         print(f"Downloading batch no {batch_no}")
#         subprocess.run(f"scp -P 2098 -r root@161.35.160.152:/mnt/volume_lon1_01/spc_backup/batch_{batch_no} {folder}")
#         if len([i for i in Path(f"{folder}").rglob(f"*{batch_no}*")]) == 0:
#             continue
        
#         print(f"Downloaded batch no {batch_no}")
        
#         #opus_df = pd.DataFrame({'opus': ["_".join(str(i.name).split("_")[:-2]) for i in Path(folder).rglob("**/*.0")]})
#         opus_df = pd.DataFrame({'opus': [(str(i.name).split("_")[0]) for i in Path(folder).rglob("**/*.0")]})
#         opus_df = opus_df.sort_values("opus")
#         opus_df = opus_df[opus_df.duplicated(subset="opus")]
#         opus_df.to_csv("tst.csv")
  
#         for file in [c for c in Path(folder).rglob(f"*{farmlab_barcode}*")]:
#             print(file,"****************")
#             renaming[barcode] = farmlab_barcode
#             converted_opus = [i for i in Path(f"outputFiles/{renamed_opus_folder}/{subfolder}").rglob(f"*{barcode}*")]
#             print(f"No of converted opus files for {barcode}: {len(converted_opus)}")
#             if(len(converted_opus) == 2):
#                break
#             directory = file.parent
#             name = file.name
#             actual_barcode = name.split("_")[0]
#             print(f"{barcode}_{len(converted_opus)}.0")
#             shutil.copyfile(file,f"outputFiles/{opus_folder}/{subfolder}/{name}")
#             os.rename(f"outputFiles/{opus_folder}/{subfolder}/{name}",f"outputFiles/{renamed_opus_folder}/{subfolder}/{barcode}_{len(converted_opus)}.0")
               
#             record['Previous Crop'] = crop
#             record['Next Crop'] = crop
#             record['Other Crops'] = crop
#             record['Barcode'] = barcode
#             try:
#                 record['Report Language'] = 'en'
#                 record['Analysis Name'] = "Starter Soil Scan (IR)"
#                 # record['Tree Age(Years)'] = row['Age']
#                 # record['Tree_Population(Total in Field)'] = row['Plant density']
#                 # record['Field Size (Acre)'] = row['Acres']
#                 if 'AEZ_name' in comparison_df.columns:                     
#                     record['Latitude'] = aez_df.loc[aez_df['AEZ_name']==aez]['latitude'].values[0]
#                     record['Longitude'] = aez_df.loc[aez_df['AEZ_name']==aez]['longitude'].values[0]
#                 else:
#                     record['Latitude'] = '-1.036'
#                     record['Longitude'] = '36.84'
#                 _ = pd.DataFrame(record).T
#                 if 'Yield Target' in comparison_df.columns and row['Yield Target']:
#                     _ = _[['Barcode','Name of Farmer','Phone Number','Sampler Name','Longitude','Latitude','Sample Date','Previous Crop','Next Crop','Other Crops',	'Report Language',	'Analysis Name']]
#                 else:
#                     _ = _[['Barcode','Name of Farmer','Phone Number','Sampler Name','Longitude','Latitude','Sample Date','Previous Crop','Next Crop','Other Crops',	'Report Language',	'Analysis Name', 'Tree_Population(Total in Field)',	'Tree Age(Years)','Field Size (Acre)'	]]
#                 spectral_sample_df = pd.concat([spectral_sample_df, _])
#                 spectral_df.to_csv("Spectral Sample Output.csv")
#                 print(f"No of converted opus files for {barcode} after: {len(converted_opus)}")
#             except Exception as e:
#                 print(e)
       

In [43]:
comparison_df.iloc[4:]

,barcode,client_name,farm_name,crop,country_name,ph,phosphorus,potassium,Organic Matter,Yield Target,SOIL CORRECTION,SOIL CORRECTION.1,SOIL CORRECTION:MANURE/COMPOST,PLANTING,PLANTING.1,PLANTING.2,PLANTING.3,TOP DRESS,TOP DRESS.1,TOP DRESS.2
4,TEST-DS3-0005,Institute for International Economic Studies,test,Maize,Uganda,optimum,optimum,optimum,optimum,5,NaN,NaN,5000.0,NaN,NaN,NaN,NPK 12.24.12 +5S,NaN,UREA,NaN
5,TEST-DS3-0006,Institute for International Economic Studies,test,Maize,Uganda,low,optimum,low,low,5,calcitic lime,dolomitic lime,5000.0,NaN,NaN,NaN,NPK 12.24.12 +5S,CAN,NaN,MOP
6,TEST-DS3-0007,Institute for International Economic Studies,test,Maize,Uganda,high,low,high,high,5,NaN,NaN,5000.0,NaN,NaN,DAP,NaN,NaN,UREA,NaN
7,TEST-DS3-0008,Institute for International Economic Studies,test,Maize,Uganda,optimum,low,optimum,optimum,5,NaN,NaN,5000.0,NaN,NaN,NaN,NPK 12.24.12 +5S,NaN,UREA,NaN
8,TEST-DS3-0009,Institute for International Economic Studies,test,Maize,Uganda,low,low,low,low,5,calcitic lime,dolomitic lime,5000.0,NaN,NaN,NaN,NPK 12.24.12 +5S,CAN,NaN,MOP


In [45]:
renamed_opus_folder = "opus_renamed"
opus_folder = "opus"
spectral_sample_output_file = "Spectral Sample Output.csv"
location = "Kenya"
# lat, lng = getCoords(location)
renaming = {}

os.makedirs(f"outputFiles/{opus_folder}",exist_ok=True)
os.makedirs(f"outputFiles/{renamed_opus_folder}",exist_ok=True)


aez_df = pd.read_csv("inputFiles/2024-08-06 Nigeria AEZ coordinates.csv")
aez_df = aez_df.reset_index()
for index, row in comparison_df.iterrows():
    count+=1
    subfolder = int(count/100)
    os.makedirs(f"outputFiles/{opus_folder}/{subfolder}",exist_ok=True)
    os.makedirs(f"outputFiles/{renamed_opus_folder}/{subfolder}",exist_ok=True)
    barcode = row['barcode'].strip()
    crop = row['crop'].strip()
    client = row['client_name'].strip()
    farm = row['farm_name'].strip()
    ph = str(row['ph']).strip()
    phosphorus = str(row['phosphorus']).strip()
    # potassium = str(row['potassium']).strip()
    organic_matter = str(row['Organic Matter']).strip()
    #calcium = str(row['calcium']).strip()
    # magnesium = str(row['magnesium']).strip()
    print(barcode)

    # folder = f"./outputFiles/{crop}_ph-{ph}_phosphorus-{phosphorus}_potassium-{potassium}_organicmatter-{organic_matter}_calcium-{calcium}_magnesium-{magnesium}"
    #folder = f"./outputFiles/{crop.replace(' ','')}_ph-{ph}_phosphorus-{phosphorus}_potassium-{potassium}_organicmatter-{organic_matter}_calcium-{calcium}"
    folder = f"./outputFiles/{crop.replace(' ','')}_ph-{ph}_phosphorus-{phosphorus}_organicmatter-{organic_matter}"
    if(folder in os.listdir("./outputFiles")):
        continue

    crop_code = cursor_lims.execute(f"""
        DECLARE @crop_code INT
        SELECT @crop_code = isnull(crop_code,0) from dbo.Crops WHERE crop_name = '{crop}'
        SELECT @crop_code
    """).fetchone()
    client_id = cursor_lims.execute(f"""
        DECLARE @client_id INT
        SELECT @client_id = isnull(client_id,0) from dbo.Clients WHERE client_name = '{client}'
        SELECT @client_id
    """).fetchone()

    ph_code = cursor_lims.execute(f"""
        DECLARE @chemical_code INT
        SELECT @chemical_code = isnull(chemical_code,0) from dbo.chemicals WHERE chemical_name = 'ph'
        SELECT @chemical_code
    """).fetchone()
    phosphorus_code = cursor_lims.execute(f"""
        DECLARE @chemical_code INT
        SELECT @chemical_code = isnull(chemical_code,0) from dbo.chemicals WHERE chemical_name = 'phosphorus'
        SELECT @chemical_code
    """).fetchone()
    potassium_code = cursor_lims.execute(f"""
        DECLARE @chemical_code INT
        SELECT @chemical_code = isnull(chemical_code,0) from dbo.chemicals WHERE chemical_name = 'potassium'
        SELECT @chemical_code
    """).fetchone()
    organic_matter_code = cursor_lims.execute(f"""
        DECLARE @chemical_code INT
        SELECT @chemical_code = isnull(chemical_code,0) from dbo.chemicals WHERE chemical_name = 'organic matter'
        SELECT @chemical_code
    """).fetchone()
    calcium_code = cursor_lims.execute(f"""
        DECLARE @chemical_code INT
        SELECT @chemical_code = isnull(chemical_code,0) from dbo.chemicals WHERE chemical_name = 'calcium'
        SELECT @chemical_code
    """).fetchone()

    try:
        aez = row['AEZ_name']
    except Exception as e:
        print(e)

    
    os.makedirs(folder, exist_ok=True)
    
    # vindexes = pd.read_sql(f"""
    #     SELECT vIndexes.guide, Clients.client_id, Clients.client_name, LOWER(vIndexes.status_name) AS status_name, Crops.crop_name, vIndexes.crop_code, LOWER(vIndexes.Chemical_Name) AS chemical_name, chemicals.chemical_code 
    #     FROM vIndexes
    #     INNER JOIN Crops
    #     ON Crops.crop_code = vIndexes.crop_code
    #     INNER JOIN Clients
    #     ON Clients.client_id = vIndexes.client_id
    #     INNER JOIN Chemicals
    #     ON Chemicals.Chemical_Name = vIndexes.Chemical_Name
    #     WHERE
    #     vIndexes.lab_code =7 and
    #     vIndexes.group_code =3 and
    #     vIndexes.growth_code =0 and
    #     vIndexes.crop_name = '{crop}'  AND
    #     Clients.client_name = '{client}' AND 
    #     Clients.client_type=6 AND
    #     guide IS NOT NULL
    #     """,con=conn_lims)
    vindexes = pd.read_sql(f"""
        SELECT vIndexes.guide, Clients.client_id, Clients.client_name, LOWER(vIndexes.status_name) AS status_name, Crops.crop_name, vIndexes.crop_code, LOWER(vIndexes.Chemical_Name) AS chemical_name, chemicals.chemical_code 
        FROM vIndexes
        INNER JOIN Crops
        ON Crops.crop_code = vIndexes.crop_code
        INNER JOIN Clients
        ON Clients.client_id = vIndexes.client_id
        INNER JOIN Chemicals
        ON Chemicals.Chemical_Name = vIndexes.Chemical_Name
        WHERE
        vIndexes.lab_code =7 and
        vIndexes.group_code =3 and
        vIndexes.growth_code =0 and
        vIndexes.crop_name = '{crop}'  AND
        Clients.client_name = '{client}' AND 
        Clients.client_type=6 AND
        guide IS NOT NULL
        """,con=conn_lims)

    if len(vindexes) == 0:
        print("No guides found for:")
        print(crop)
        print(client)
        print("*******************************************")
        continue

    


        
    vindexes.to_csv(f"{folder}/vindexes.csv")
    vindexes.chemical_name =  [str(i).replace(" ","_") for i in vindexes.chemical_name]
    vindexes.chemical_name = [i.strip().replace(" ","_").replace(".","").replace("(","").replace(")","") for i in vindexes.chemical_name]
    vindexes = vindexes.drop_duplicates()
    #calcium_guides = vindexes.loc[vindexes['chemical_name']=='calcium']
    #calcium_guides = calcium_guides.drop_duplicates(subset='status_name')
    #calcium_guides = calcium_guides[['status_name','guide']]
    #calcium_guides = calcium_guides.set_index('status_name')
    #calcium_guides = calcium_guides.to_dict()
    #print("Calcium guides:", calcium_guides)
    
    # magnesium_guides = vindexes.loc[vindexes['chemical_name']=='magnesium']
    # magnesium_guides = magnesium_guides.drop_duplicates(subset='status_name')
    # magnesium_guides = magnesium_guides[['status_name','guide']]
    # magnesium_guides = magnesium_guides.set_index('status_name')
    # magnesium_guides = magnesium_guides.to_dict()

    # potassium_guides = vindexes.loc[vindexes['chemical_name']=='potassium']
    # potassium_guides = potassium_guides.drop_duplicates(subset='status_name')
    # potassium_guides = potassium_guides[['status_name','guide']]
    # potassium_guides = potassium_guides.set_index('status_name')
    # potassium_guides = potassium_guides.to_dict()

    organic_matter_guides = vindexes.loc[vindexes['chemical_name']=='organic_matter']
    organic_matter_guides = organic_matter_guides.drop_duplicates(subset='status_name')
    organic_matter_guides = organic_matter_guides[['status_name','guide']]
    organic_matter_guides = organic_matter_guides.set_index('status_name')
    organic_matter_guides = organic_matter_guides.to_dict()
    print("OM guides:", organic_matter_guides)
    
    ph_guides = vindexes.loc[vindexes['chemical_name']=='ph']
    ph_guides = ph_guides.drop_duplicates(subset='status_name')
    ph_guides = ph_guides[['status_name','guide']]
    ph_guides = ph_guides.set_index('status_name')
    ph_guides = ph_guides.to_dict()
    print("ph guides:", ph_guides)

    crop_reports = scores.copy(deep=True)

    # guides = pd.DataFrame()
    # ph_guides_ = crop_reports[['ph','cecCalcul']].apply(lambda row: convertGuides(row, chemical="ph", verylow=ph_guides['guide']['very low'], low=ph_guides['guide']['low'], optimum=ph_guides['guide']['critical'], high=ph_guides['guide']['high'],  veryhigh=ph_guides['guide']['very high'], crop_code=crop_code[0], client_id=client_id[0], chemical_code=ph_code[0]), axis=1)                                              
    # om_guides_ = crop_reports[['organic_matter','cecCalcul']].apply(lambda row: convertGuides(row, chemical="organic_matter", verylow=organic_matter_guides['guide']['very low'], low=organic_matter_guides['guide']['low'], optimum=organic_matter_guides['guide']['critical'], high=organic_matter_guides['guide']['high'],  veryhigh=organic_matter_guides['guide']['very high'], crop_code=crop_code[0], client_id=client_id[0], chemical_code=organic_matter_code[0]), axis=1)                                              
    # #crop_reports['calcium_classes'] = crop_reports[['calciumPercSat','cecCalcul']].apply(lambda row: classifyResults(row, chemical="calciumPercSat", lim1=calcium_guides['guide']['very low'], lim2=calcium_guides['guide']['low'], lim3=calcium_guides['guide']['high'], crop_code=crop_code, client_id=client_id, chemical_code=calcium_code), axis=1)
    # # crop_reports['magnesium'] = crop_reports[['magnesiumPercSat','cecCalcul']].apply(lambda row: classifyResults(row, chemical="magnesiumPercSat", lim1=magnesium_guides['guide']['very low'], lim2=magnesium_guides['guide']['low'], lim3=magnesium_guides['guide']['high'], crop_code=crop_code, client_id=client_id, chemical_code=magnesium_code), axis=1)
    # potassium_guides_ = crop_reports[['potassium','cecCalcul']].apply(lambda row: convertGuides(row, chemical="potassium", verylow=potassium_guides['guide']['very low'], low=potassium_guides['guide']['low'], optimum=potassium_guides['guide']['critical'], high=potassium_guides['guide']['high'],  veryhigh=potassium_guides['guide']['very high'], crop_code=crop_code[0], client_id=client_id[0], chemical_code=potassium_code[0]), axis=1)
    # guides = pd.concat([ph_guides_, om_guides_, potassium_guides_])
    # guides.to_csv(f"{folder}/guides.csv")

    print("Details: ",crop_code[0], client_id[0], organic_matter_code[0])
    crop_reports = crop_reports.loc[:, ~crop_reports.columns.duplicated()]
    print(crop_reports.columns)

    crop_reports['phosphorus_classes'] = crop_reports['phosphorus']
    if(phosphorus in ['low','very low','optimum','high','very high']):
        crop_reports = crop_reports.loc[crop_reports['phosphorus_classes'] == phosphorus]
        print("phosphorus class: ",phosphorus)
        print("phosphorus",len(crop_reports))
        
    crop_reports['ph_classes'] = crop_reports[['ph','cecCalcul']].apply(lambda row: classifyResults(row, chemical="ph", lim1=ph_guides['guide']['very low'], lim2=ph_guides['guide']['low'], lim3=ph_guides['guide']['high'], crop_code=crop_code[0], client_id=client_id[0], chemical_code=ph_code[0]), axis=1)                                              
    if(ph in ['low','very low','optimum','high','very high']):
        crop_reports = crop_reports.loc[crop_reports['ph_classes'] == ph]
        print("ph class: ",ph)
        print("ph",len(crop_reports)) 

    crop_reports['organic_matter_classes'] = np.nan
    if(organic_matter in ['low','very low','optimum','high','very high']):
        crop_reports['organic_matter_classes'] = crop_reports[['organic_matter','cecCalcul']].apply(lambda row: classifyResults(row, chemical="organic_matter", lim1=organic_matter_guides['guide']['very low'], lim2=organic_matter_guides['guide']['low'], lim3=organic_matter_guides['guide']['high'], crop_code=crop_code[0], client_id=client_id[0], chemical_code=organic_matter_code[0]), axis=1)                                                  
        crop_reports = crop_reports.loc[crop_reports['organic_matter_classes'] == organic_matter]
        print("organic matter class: ",organic_matter)
        print("organic_matter",len(crop_reports))

    
    # crop_reports['potassium_classes'] = np.nan
    # if len(crop_reports) > 0:
    #     if(potassium in ['low','very low','optimum','high','very high']):
    #         crop_reports['potassium_classes'] = crop_reports[['potassium','cecCalcul']].apply(lambda row: classifyResults(row, chemical="potassium", lim1=potassium_guides['guide']['very low'], lim2=potassium_guides['guide']['low'], lim3=potassium_guides['guide']['high'], crop_code=crop_code[0], client_id=client_id[0], chemical_code=potassium_code[0]), axis=1)
    #         crop_reports = crop_reports.loc[crop_reports['potassium_classes'] == potassium]
    #         print("potassium class: ",potassium)
    #         print("potassium",len(crop_reports))
        
    #crop_reports['calcium_classes'] = crop_reports[['calciumPercSat','cecCalcul']].apply(lambda row: classifyResults(row, chemical="calciumPercSat", lim1=calcium_guides['guide']['very low'], lim2=calcium_guides['guide']['low'], lim3=calcium_guides['guide']['high'], crop_code=crop_code, client_id=client_id, chemical_code=calcium_code), axis=1)
    #crop_reports['magnesium'] = crop_reports[['magnesiumPercSat','cecCalcul']].apply(lambda row: classifyResults(row, chemical="magnesiumPercSat", lim1=magnesium_guides['guide']['very low'], lim2=magnesium_guides['guide']['low'], lim3=magnesium_guides['guide']['high'], crop_code=crop_code, client_id=client_id, chemical_code=magnesium_code), axis=1)

    crop_reports[['spectral_sample_id','ph','ph_classes','phosphorus','phosphorus_classes','organic_matter','organic_matter_classes']].to_csv(f"{folder}/crop_reports.csv")
    crop_reports[['spectral_sample_id','ph','potassium','organic_matter']].set_index("spectral_sample_id").describe().to_csv(f"{folder}/per_parameter_summary_stats.csv")
    crop_reports[['spectral_sample_id','ph_classes','phosphorus_classes','organic_matter_classes']].set_index("spectral_sample_id").value_counts().to_csv(f"{folder}/soil_classes_count.csv")
    
    

    # crop_reports[['spectral_sample_id','ph','ph_classes','potassium','potassiumPercSat','potassium_classes','phosphorus','phosphorus_classes','organic_matter','organic_matter_classes']].to_csv(f"{folder}/crop_reports_final.csv")
    crop_reports[['spectral_sample_id','ph','ph_classes','phosphorus','phosphorus_classes','organic_matter','organic_matter_classes']].to_csv(f"{folder}/crop_reports_final.csv")

    
    if(len(crop_reports) == 0):
        continue
    client_id = vindexes.client_id.unique()[0]
    crop_code = vindexes.crop_code.unique()[0]
    
    spectral_df = pd.read_sql(f"""
        SELECT 
        spectral_sample_id, 
        barcode AS 'Barcode', 
        farmer_name AS 'Name of Farmer', 
        phone_number AS 'Phone Number',
        sampler_name AS 'Sampler Name', 
        spectral_batch_id, 
        longitude AS 'Longitude',
        latitude AS 'Latitude',
        sample_date AS 'Sample Date', 
        analysis_type AS 'Analysis Name', 
        tree_population AS 'Tree_Population(Total in Field)',  
        TIMESTAMPDIFF(YEAR,'2024-05-28',date_of_planting) AS 'Tree Age(Years)', 
        field_size AS 	'Field Size (Acre)'
        FROM SpectralSample
        WHERE spectral_sample_id IN {str([str(j) for j in crop_reports['spectral_sample_id'].values]).replace("[","(").replace("]",")")} 
        AND spectral_batch_id > 3355
        AND batch_date > '2023-06-01'
    """, con=conn)
    # scoring_res = pd.read_sql("
    for index, record in spectral_df.iterrows():
        farmlab_barcode = record['Barcode']
        batch_no = record['spectral_batch_id']
        batch_no = str(batch_no).split('.')[0]
        
        if len([i for i in Path(f'./outputFiles/{renamed_opus_folder}/{subfolder}').rglob(f"*{barcode}*")]) == 2:
            break
        #try:
        print(f"Downloading batch no {batch_no}")
        subprocess.run(f"scp -P 2098 -r root@161.35.160.152:/mnt/volume_lon1_01/spc_backup/batch_{batch_no} {folder}")
        if len([i for i in Path(f"{folder}").rglob(f"*{batch_no}*")]) == 0:
            continue
        
        print(f"Downloaded batch no {batch_no}")
        
        #opus_df = pd.DataFrame({'opus': ["_".join(str(i.name).split("_")[:-2]) for i in Path(folder).rglob("**/*.0")]})
        opus_df = pd.DataFrame({'opus': [(str(i.name).split("_")[0]) for i in Path(folder).rglob("**/*.0")]})
        opus_df = opus_df.sort_values("opus")
        opus_df = opus_df[opus_df.duplicated(subset="opus")]
        opus_df.to_csv(f"{folder}/opus_barcodes.csv")
  
        for file in [c for c in Path(folder).rglob(f"*{farmlab_barcode}*")]:
            print(file,"****************")
            renaming[barcode] = farmlab_barcode
            converted_opus = [i for i in Path(f"outputFiles/{renamed_opus_folder}/{subfolder}").rglob(f"*{barcode}*")]
            print(f"No of converted opus files for {barcode}: {len(converted_opus)}")
            if(len(converted_opus) == 2):
               break
            directory = file.parent
            name = file.name
            actual_barcode = name.split("_")[0]
            print(f"{barcode}_{len(converted_opus)}.0")
            shutil.copyfile(file,f"outputFiles/{opus_folder}/{subfolder}/{name}")
            os.rename(f"outputFiles/{opus_folder}/{subfolder}/{name}",f"outputFiles/{renamed_opus_folder}/{subfolder}/{barcode}_{len(converted_opus)}.0")
               
            record['Previous Crop'] = crop
            record['Next Crop'] = crop
            record['Other Crops'] = crop
            record['Barcode'] = barcode
            try:
                record['Report Language'] = 'en'
                record['Analysis Name'] = "Starter Soil Scan (IR)"
                # record['Tree Age(Years)'] = row['Age']
                # record['Tree_Population(Total in Field)'] = row['Plant density']
                # record['Field Size (Acre)'] = row['Acres']
                if 'AEZ_name' in comparison_df.columns:                     
                    record['Latitude'] = aez_df.loc[aez_df['AEZ_name']==aez]['latitude'].values[0]
                    record['Longitude'] = aez_df.loc[aez_df['AEZ_name']==aez]['longitude'].values[0]
                else:
                    record['Latitude'] = '-1.036'
                    record['Longitude'] = '36.84'
                _ = pd.DataFrame(record).T
                if 'Yield Target' in comparison_df.columns and row['Yield Target']:
                    _ = _[['Barcode','Name of Farmer','Phone Number','Sampler Name','Longitude','Latitude','Sample Date','Previous Crop','Next Crop','Other Crops',	'Report Language',	'Analysis Name']]
                else:
                    _ = _[['Barcode','Name of Farmer','Phone Number','Sampler Name','Longitude','Latitude','Sample Date','Previous Crop','Next Crop','Other Crops',	'Report Language',	'Analysis Name', 'Tree_Population(Total in Field)',	'Tree Age(Years)','Field Size (Acre)'	]]
                spectral_sample_df = pd.concat([spectral_sample_df, _])
                spectral_df.to_csv("Spectral Sample Output.csv")
                print(f"No of converted opus files for {barcode} after: {len(converted_opus)}")
            except Exception as e:
                print(e)
       

TEST-DS3-0001
'AEZ_name'


C:\Users\tsuma.thomas\AppData\Local\Temp\ipykernel_18412\1982317046.py:100: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  vindexes = pd.read_sql(f"""


OM guides: {'guide': {'low': 2.5, 'high': 8.0, 'very high': 10.0, 'critical': 5.0, 'very low': 1.5}}
ph guides: {'guide': {'low': 5.4, 'high': 7.0, 'very high': 7.5, 'critical': 6.4, 'very low': 5.0}}
Details:  46 64540 17
Index(['Unnamed: 0.1', 'Unnamed: 0', 'spectral_sample_id', 'sample_date',
       'spectral_batch_id', 'id', 'spectral_sample_id.1',
       'spectral_batch_id.1', 'lab_id', 'phosphorus', 'texture', 'aluminium',
       'boron', 'calcium', 'clay', 'copper', 'ec_salts',
       'exchangeable_acidity', 'iron', 'magnesium', 'manganese',
       'organic_carbon', 'ph', 'phosphorus_sorption_index', 'potassium',
       'sand', 'silt', 'sodium', 'sulphur', 'total_nitrogen', 'zinc',
       'cn_ratio', 'acid_saturation', 'hydrogen_percent', 'other_bases',
       'cecCalcul', 'calciumPercSat', 'magnesiumPercSat', 'potassiumPercSat',
       'sodiumPercSat', 'CaMgRatio', 'CNRatio', 'organic_matter'],
      dtype='object')
phosphorus class:  high
phosphorus 1956
ph class:  high
ph 885

C:\Users\tsuma.thomas\AppData\Local\Temp\ipykernel_18412\1982317046.py:227: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  spectral_df = pd.read_sql(f"""


Downloaded batch no 7600
outputFiles\Maize_ph-high_phosphorus-high_organicmatter-high\batch_7600\20230814160810\ocp-gh-brr201_20230814_141029_001.0 ****************
No of converted opus files for TEST-DS3-0001: 0
TEST-DS3-0001_0.0
No of converted opus files for TEST-DS3-0001 after: 0
outputFiles\Maize_ph-high_phosphorus-high_organicmatter-high\batch_7600\20230814160810\ocp-gh-brr201_20230814_141105_002.0 ****************
No of converted opus files for TEST-DS3-0001: 1
TEST-DS3-0001_1.0
No of converted opus files for TEST-DS3-0001 after: 1
TEST-DS3-0002
'AEZ_name'


...

Downloaded batch no 7592
outputFiles\Maize_ph-optimum_phosphorus-low_organicmatter-optimum\batch_7592\20230802200853\OCP-GH-BR649_20230802_163742_001.0 ****************
No of converted opus files for TEST-DS3-0008: 0
TEST-DS3-0008_0.0
No of converted opus files for TEST-DS3-0008 after: 0
outputFiles\Maize_ph-optimum_phosphorus-low_organicmatter-optimum\batch_7592\20230802200853\OCP-GH-BR649_20230802_163816_002.0 ****************
No of converted opus files for TEST-DS3-0008: 1
TEST-DS3-0008_1.0
No of converted opus files for TEST-DS3-0008 after: 1
TEST-DS3-0009
'AEZ_name'


C:\Users\tsuma.thomas\AppData\Local\Temp\ipykernel_18412\1982317046.py:100: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  vindexes = pd.read_sql(f"""


OM guides: {'guide': {'low': 2.5, 'high': 8.0, 'very high': 10.0, 'critical': 5.0, 'very low': 1.5}}
ph guides: {'guide': {'low': 5.4, 'high': 7.0, 'very high': 7.5, 'critical': 6.4, 'very low': 5.0}}
Details:  46 64540 17
Index(['Unnamed: 0.1', 'Unnamed: 0', 'spectral_sample_id', 'sample_date',
       'spectral_batch_id', 'id', 'spectral_sample_id.1',
       'spectral_batch_id.1', 'lab_id', 'phosphorus', 'texture', 'aluminium',
       'boron', 'calcium', 'clay', 'copper', 'ec_salts',
       'exchangeable_acidity', 'iron', 'magnesium', 'manganese',
       'organic_carbon', 'ph', 'phosphorus_sorption_index', 'potassium',
       'sand', 'silt', 'sodium', 'sulphur', 'total_nitrogen', 'zinc',
       'cn_ratio', 'acid_saturation', 'hydrogen_percent', 'other_bases',
       'cecCalcul', 'calciumPercSat', 'magnesiumPercSat', 'potassiumPercSat',
       'sodiumPercSat', 'CaMgRatio', 'CNRatio', 'organic_matter'],
      dtype='object')
phosphorus class:  low
phosphorus 8211
ph class:  low
ph 538
o

C:\Users\tsuma.thomas\AppData\Local\Temp\ipykernel_18412\1982317046.py:227: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  spectral_df = pd.read_sql(f"""


Downloaded batch no 7995
outputFiles\Maize_ph-low_phosphorus-low_organicmatter-low\batch_7995\20230905160908\OCP-T2-20873_20230905_163712_001.0 ****************
No of converted opus files for TEST-DS3-0009: 0
TEST-DS3-0009_0.0
No of converted opus files for TEST-DS3-0009 after: 0
outputFiles\Maize_ph-low_phosphorus-low_organicmatter-low\batch_7995\20230905160908\OCP-T2-20873_20230905_163746_002.0 ****************
No of converted opus files for TEST-DS3-0009: 1
TEST-DS3-0009_1.0
No of converted opus files for TEST-DS3-0009 after: 1


In [46]:
comparison_df.tail()

,barcode,client_name,farm_name,crop,country_name,ph,phosphorus,potassium,Organic Matter,Yield Target,SOIL CORRECTION,SOIL CORRECTION.1,SOIL CORRECTION:MANURE/COMPOST,PLANTING,PLANTING.1,PLANTING.2,PLANTING.3,TOP DRESS,TOP DRESS.1,TOP DRESS.2
4,TEST-DS3-0005,Institute for International Economic Studies,test,Maize,Uganda,optimum,optimum,optimum,optimum,5,NaN,NaN,5000.0,NaN,NaN,NaN,NPK 12.24.12 +5S,NaN,UREA,NaN
5,TEST-DS3-0006,Institute for International Economic Studies,test,Maize,Uganda,low,optimum,low,low,5,calcitic lime,dolomitic lime,5000.0,NaN,NaN,NaN,NPK 12.24.12 +5S,CAN,NaN,MOP
6,TEST-DS3-0007,Institute for International Economic Studies,test,Maize,Uganda,high,low,high,high,5,NaN,NaN,5000.0,NaN,NaN,DAP,NaN,NaN,UREA,NaN
7,TEST-DS3-0008,Institute for International Economic Studies,test,Maize,Uganda,optimum,low,optimum,optimum,5,NaN,NaN,5000.0,NaN,NaN,NaN,NPK 12.24.12 +5S,NaN,UREA,NaN
8,TEST-DS3-0009,Institute for International Economic Studies,test,Maize,Uganda,low,low,low,low,5,calcitic lime,dolomitic lime,5000.0,NaN,NaN,NaN,NPK 12.24.12 +5S,CAN,NaN,MOP


In [47]:
spectral_sample_df = spectral_sample_df.reset_index().drop_duplicates(subset="Barcode").set_index("Barcode")
start = 0
count = 0
import  math
maximum = math.ceil((len(comparison_df)+100) / 100) * 100
for i in (np.arange(0,maximum,100)):
    print(start,"   ",i)
    if start == i:
        continue
    _ =  spectral_sample_df.loc[spectral_sample_df.index.isin(comparison_df.iloc[start:i]['barcode'])]
    start = i
    _.to_excel(f"outputFiles/{renamed_opus_folder}/{count}/Spectral Sample Output.xlsx")
    count+=1

0     0
0     100


In [48]:
# !pip install openpyxl

In [49]:
with open('./outputFiles/renaming.json', 'w') as outfile:
    json.dump(renaming, outfile)
subprocess.run(f"powershell Compress-Archive {os.getcwd()}\\outputFiles\\{renamed_opus_folder} {os.getcwd()}\\outputFiles\\opus.zip")
# spectral_sample_df = spectral_sample_df.drop_duplicates(subset="Barcode").set_index("Barcode")
spectral_sample_df.to_excel("outputFiles/Spectral Sample Output.xlsx")

In [ ]:
spectral_sample_df

In [ ]:
aez_df = pd.read_csv("inputFiles/2024-06-17 links between AEZ present in Kenya and coordinates.csv")
aez_df = aez_df.reset_index()

In [ ]:
aez_df
for index, row in aez_df.iterrows():
    lat = row['latitude']
    lng = row['longitude']
    name = row['AEZ_name']
    print(name)
    print(spectral_sample_df.loc[spectral_sample_df['Latitude'] == name])
    spectral_sample_df.loc[spectral_sample_df['Latitude'] == name, 'Latitude'] = lat
    spectral_sample_df.loc[spectral_sample_df['Longitude'] == name, 'Longitude'] = lng

In [ ]:
spectral_sample_df.to_csv("outputFiles/Spectral Sample Output.csv")

In [ ]:
vindexes

In [ ]:
comparison_df

In [ ]:
scores.loc[scores['spectral_sample_id'].isin(crop_reports['spectral_sample_id'])]['ph']

In [ ]:
crop_reports

In [ ]:
comparison_df

In [ ]:
cl = pd.read_sql("SELECT TOP(10) * FROM Clients WHERE client_name = 'FMAFS-National'",con=conn_lims)

In [ ]:
cl

In [ ]:
v = pd.read_sql("SELECT * FROM vIndexes where vIndexes.crop_name LIKE '%Sorghum (hybrid)%' AND chemical_name IN ('ph','organic matter','calcium') AND client_id = 59794",con=conn_lims)

In [ ]:
v

In [ ]:
c = pd.read_sql("SELECT * FROM Chemicals WHERE chemical_name LIKE '%pot%'",con=conn_lims)

In [ ]:
cr = pd.read_sql("SELECT TOP(1)* FROM vIndexes WHERE crop_name LIKE '%Sorghum (hybrid)%' AND client_id = 59789" ,con=conn_lims)

In [ ]:
cr

In [ ]:
sr = pd.read_sql("SELECT * FROM ScoringResult sr WHERE sr.chemical_id = 36 LIMIT 10",con=conn)

In [ ]:
sr

In [ ]:
c = pd.read_sql("SELECT * FROM MeasuredChemical WHERE chemical_name LIKE '%pot%'",con=conn)

In [ ]:
c

In [ ]:
crop_reports

In [ ]:
vindexes = pd.read_sql(f"""
        SELECT vIndexes.guide, Clients.client_id, Clients.client_name, LOWER(vIndexes.status_name) AS status_name, Crops.crop_name, vIndexes.crop_code, LOWER(vIndexes.Chemical_Name) AS chemical_name 
        FROM vIndexes 
        INNER JOIN Crops 
        ON Crops.crop_code = vIndexes.crop_code
        INNER JOIN Clients
        ON Clients.client_id = vIndexes.client_id
        WHERE 
        vIndexes.crop_name LIKE '%Millet%' AND 
        Clients.client_name LIKE '%OCP Kenya%' AND
        guide IS NOT NULL
    """,con=conn_lims)

In [ ]:
vindexes.loc[vindexes['chemical_name']=="potassium"]

In [ ]:
vindexes.chemical_name =  [str(i).replace(" ","_") for i in vindexes.chemical_name]
vindexes.chemical_name = [i.strip().replace(" ","_").replace(".","").replace("(","").replace(")","") for i in vindexes.chemical_name]
vindexes = vindexes.drop_duplicates()

In [ ]:
vindexes

In [ ]:
organic_matter_guides = vindexes.loc[vindexes['chemical_name']=='organic_matter']
organic_matter_guides = organic_matter_guides.drop_duplicates(subset='status_name')
organic_matter_guides = organic_matter_guides[['status_name','guide']]
organic_matter_guides = organic_matter_guides.set_index('status_name')
organic_matter_guides = organic_matter_guides.to_dict()

In [ ]:
organic_matter_guides

In [ ]:
scores.loc[scores['spectral_sample_id'] == 112530]['organic_matter']

In [ ]:
scores.head()

In [ ]:
crops = pd.read_sql("SELECT * FROM Crops WHERE crop_name LIKE '%Passion%'",con=conn_lims)

In [ ]:
crops

In [ ]:
chemicals

In [ ]:
output

In [ ]:
vindexes = pd.read_sql(f"""
        SELECT vIndexes.*,  vIndexes.guide, Clients.client_id, Clients.client_name, LOWER(vIndexes.status_name) AS status_name, Crops.crop_name, vIndexes.crop_code, LOWER(vIndexes.Chemical_Name) AS chemical_name, chemicals.chemical_code
        FROM vIndexes
        INNER JOIN Crops
        ON Crops.crop_code = vIndexes.crop_code
        INNER JOIN Clients
        ON Clients.client_id = vIndexes.client_id
        INNER JOIN Chemicals
        ON Chemicals.Chemical_Name = vIndexes.Chemical_Name
        WHERE
        vIndexes.lab_code =7 and
        vIndexes.group_code =3 and
        vIndexes.growth_code =0 and
        vIndexes.crop_name IN ('Maize','Coffee','Cabbage','Beans','Tomatoes (Open field)','Kale','Tea','Sugar Cane','Potatoes (Irish)','Sunflower','Soya') AND
        Clients.client_name = 'Kalro' AND  Clients.client_type=6 AND
        guide IS NOT NULL
        """,con=conn_lims)

In [ ]:
vindexes

In [ ]:
vindexes.to_csv("Different Crops Guides for Kalro.csv")

In [ ]:
vindexes.client_id.unique()